# MuseTalk 1.5 Video Dubbing Backend on Kaggle

This notebook provisions and runs the GPU half of the video dubbing pipeline. It accepts a source video and driving audio, normalizes both inputs, invokes MuseTalk 1.5 on one Kaggle T4 GPU, restores the source resolution, and returns the generated MP4 through FastAPI.

The notebook is operational and self-contained. The MuseTalk source it invokes is tracked under `MuseTalk/` so model initialization and inference internals can be inspected directly.

## Runtime architecture

```text
Local Gradio UI
    |
    | POST /generate (video + audio)
    v
FastAPI on Kaggle
    |
    +-- ffmpeg: letterbox video to 512x512
    +-- ffmpeg: convert audio to 16 kHz mono WAV
    +-- write MuseTalk inference YAML
    v
python -m scripts.inference
    |
    +-- initialize VAE, UNet and positional encoding
    +-- initialize Whisper audio encoder
    +-- initialize DWPose, face detection and face parsing
    +-- generate and blend lip-synced frames
    v
ffmpeg: restore source resolution and return MP4
```

## Before you run
1. In Kaggle Notebook Settings, enable a GPU accelerator. A T4 instance is sufficient.
2. Attach the private Kaggle Dataset that contains the MuseTalk model bundle.
3. Set `MODEL_ROOT_PATH` and `NGROK_AUTHTOKEN` in the configuration cell.
4. Run every cell from top to bottom.
5. Keep the final cell running while the local Gradio interface sends requests.

## Notebook stages
| Stage | Responsibility |
| --- | --- |
| Configuration | Runtime paths, batch size, frame rate, server port and ngrok token |
| GPU selection | Restrict PyTorch and MuseTalk to one visible CUDA device |
| Source setup | Clone the upstream MuseTalk implementation |
| Dependencies | Install the OpenMMLab, media and API packages required on Kaggle |
| Compatibility | Adapt legacy checkpoint loading for the Kaggle PyTorch version |
| Model bundle | Link the private Dataset into `MuseTalk/models/` and validate checkpoints |
| Input processing | Generate `sanitizer.py` for deterministic video/audio conversion |
| API service | Generate `app.py` with `/health` and `/generate` endpoints |
| Public endpoint | Start Uvicorn and expose it through ngrok |

## Expected private model bundle
```
musetalkV15/
  musetalk.json
  unet.pth
syncnet/
  latentsync_syncnet.pt
dwpose/
  dw-ll_ucoco_384.pth
face-parse-bisent/
  79999_iter.pth
  resnet18-5c106cde.pth
sd-vae/
  config.json
  diffusion_pytorch_model.bin
whisper/
  config.json
  preprocessor_config.json
  pytorch_model.bin
```

`syncnet/latentsync_syncnet.pt` is retained in the model bundle for completeness and related MuseTalk workflows. The `scripts.inference` entrypoint used here does not initialize SyncNet during generation.

In [ ]:
# ── STEP 1: Configuration ─────────────────────────────────────────────────────
# Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

SERVER_PORT = 8000
BATCH_SIZE  = 8      # reduce to 4 if you hit OOM
FPS         = 25
RESULT_DIR  = "/kaggle/working/results"
MODEL_ROOT_PATH = "/kaggle/input/datasets/saisatyamjena/musetalk1-5-components/kaggle_to_upload"

print("Configuration loaded.")

In [ ]:
# ── STEP 2: Lock to single GPU (cuda:0) ──────────────────────────────────────
import os

# Must be set BEFORE any torch import
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

assert torch.cuda.is_available(), "No CUDA GPU found. Enable GPU in Notebook Settings."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
print(f"Visible devices: {torch.cuda.device_count()} (pinned to index 0)")

In [ ]:
# ── STEP 3: Clone MuseTalk source ─────────────────────────────────────────────
# --depth 1 fetches only the latest snapshot with no git history.
# This is faster and avoids network EOF errors on Kaggle.
# It has NO effect on the model weights which come from your attached dataset.
import subprocess, pathlib

MUSETALK_DIR = pathlib.Path("/kaggle/working/MuseTalk")

if not MUSETALK_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/TMElyralab/MuseTalk.git",
         str(MUSETALK_DIR)],
        check=True
    )
    print("MuseTalk cloned successfully.")
else:
    print("MuseTalk already present — skipping clone.")

print("\nTop-level contents:")
for p in sorted(MUSETALK_DIR.iterdir()):
    print(f"  {p.name}")

In [ ]:
# ── STEP 4: Install dependencies ──────────────────────────────────────────────
import subprocess, sys, pathlib

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=False)

# Fix opencv — headless build is numpy 2.x compatible
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "opencv-python", "opencv-python-headless"], capture_output=True)
pip("opencv-python-headless")

# Packages missing from Kaggle's base image
pip("soundfile")
pip("librosa")
pip("einops")
pip("omegaconf")
pip("ffmpeg-python")
pip("moviepy")
pip("imageio[ffmpeg]")
pip("gdown")
pip("pyngrok")

# OpenMMLab stack
import torch
print(f"Env — PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}")
pip("mmengine")
pip("--extra-index-url", "https://miropsota.github.io/torch_packages_builder",
    "mmcv==2.2.0+a8073c7pt2.10.0cu128")
pip("mmdet")
pip("mmpose")

# ── Patch 1: mmdet/__init__.py ──────────────────────────────────────────────
# mmdet asserts mmcv<2.2.0. The only wheel for PT2.10+cu128 is mmcv 2.2.0.
# Remove the 4-line assert block by scanning line-by-line.
mmdet_init = pathlib.Path("/usr/local/lib/python3.12/dist-packages/mmdet/__init__.py")
lines = mmdet_init.read_text().splitlines()
result, i = [], 0
while i < len(lines):
    if "assert (mmcv_version >= digit_version(mmcv_minimum_version)" in lines[i]:
        result.append("pass  # mmcv version check disabled for PT2.10+cu128")
        i += 4  # skip the 4-line assert block
    else:
        result.append(lines[i])
        i += 1
mmdet_init.write_text("\n".join(result) + "\n")
print(f"Patched mmdet: {mmdet_init}")

# ── Patch 2: mmpose hybrid_heads/__init__.py ─────────────────────────────────
# rtmo_head.py imports mmdet at module level; stub it so the import never runs.
p = pathlib.Path("/usr/local/lib/python3.12/dist-packages/mmpose/models/heads/hybrid_heads/__init__.py")
txt = p.read_text()
if "from .rtmo_head import RTMOHead" in txt:
    txt = txt.replace(
        "from .rtmo_head import RTMOHead",
        "class RTMOHead: pass  # stubbed: rtmo_head imports mmdet"
    )
    p.write_text(txt)
    print(f"Patched RTMOHead stub: {p}")

# Sanity check
import importlib, numpy as np
import cv2; importlib.reload(cv2)
import diffusers, accelerate, transformers
from accelerate.utils.memory import clear_device_cache
from diffusers import AutoencoderKL
from mmpose.apis import inference_topdown, init_model
print(f"numpy        : {np.__version__}")
print(f"cv2          : {cv2.__version__}")
print(f"diffusers    : {diffusers.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"transformers : {transformers.__version__}")
print("AutoencoderKL import  : OK")
print("clear_device_cache    : OK")
print("mmpose import         : OK")
print("\nAll checks passed.")

## Kaggle compatibility adjustments

Step 5 below changes only the temporary Kaggle environment. It does not require repository patch files.

Kaggle currently provides a newer PyTorch runtime where `torch.load` defaults to `weights_only=True`. Several MuseTalk and OpenMMLab checkpoints contain legacy serialized objects and therefore require `weights_only=False`. The cell updates affected calls inside the temporary `/kaggle/working/MuseTalk` checkout and installed `mmengine` package.

The earlier OpenMMLab adjustments address the specific MMCV/MMDetection versions available for Kaggle's CUDA and PyTorch combination. These files disappear when the Kaggle session ends.

In [ ]:
# ── STEP 5: Enable legacy checkpoint loading ──────────────────────────────────
# PyTorch 2.6+ defaults weights_only=True, breaking legacy .pth/.tar checkpoints.
# Patch every Python file under MuseTalk and mmengine that calls torch.load
# without weights_only, so inference doesn't crash on any model file.
import pathlib, re

PATCH_DIRS = [
    pathlib.Path("/kaggle/working/MuseTalk"),
    pathlib.Path("/usr/local/lib/python3.12/dist-packages/mmengine"),
]

patched_files = []
for root in PATCH_DIRS:
    if not root.exists():
        print(f"Skipping (not found): {root}")
        continue
    for py in root.rglob("*.py"):
        txt = py.read_text(errors="replace")
        # Match torch.load(...) calls that don't already have weights_only
        # Replace closing ) of torch.load(args) with , weights_only=False)
        new_txt = re.sub(
            r'torch\.load\(([^)]*?)\)',
            lambda m: (
                m.group(0) if "weights_only" in m.group(0)
                else f"torch.load({m.group(1)}, weights_only=False)"
            ),
            txt,
        )
        if new_txt != txt:
            py.write_text(new_txt)
            patched_files.append(str(py))

if patched_files:
    print(f"Patched {len(patched_files)} files:")
    for f in patched_files:
        print(f"  {f}")
else:
    print("No files needed patching.")

In [ ]:
# ── STEP 6: Link model weights from the attached Kaggle Dataset ───────────────
import shutil, pathlib

MODEL_ROOT    = pathlib.Path(MODEL_ROOT_PATH)
MODELS_TARGET = MUSETALK_DIR / "models"

if not MODEL_ROOT.exists():
    raise RuntimeError(f"Model root not found at {MODEL_ROOT}")

print(f"Model root : {MODEL_ROOT}")
print(f"Target     : {MODELS_TARGET}\n")
print("Contents of model root:")
for p in sorted(MODEL_ROOT.iterdir()):
    print(f"  {p.name}/")

MODELS_TARGET.mkdir(parents=True, exist_ok=True)

def symlink_or_copy(src: pathlib.Path, dst: pathlib.Path):
    if dst.exists() or dst.is_symlink():
        print(f"  already exists, skipping : {dst.name}")
        return
    try:
        dst.symlink_to(src)
        print(f"  symlinked : {src.name}")
    except OSError:
        shutil.copytree(str(src), str(dst))
        print(f"  copied    : {src.name}  (symlink failed)")

print("\nLinking into MuseTalk/models/:")
for sub in sorted(MODEL_ROOT.iterdir()):
    if sub.is_dir():
        symlink_or_copy(sub, MODELS_TARGET / sub.name)

print("\nMuseTalk/models/ contents:")
for p in sorted(MODELS_TARGET.iterdir()):
    print(f"  {p.name}/")

In [ ]:
# ── STEP 7: Verify all required model files ───────────────────────────────────
# MODELS_TARGET is set by the previous step; no repeated path configuration.

REQUIRED = [
    MODELS_TARGET / "musetalkV15"       / "musetalk.json",
    MODELS_TARGET / "musetalkV15"       / "unet.pth",
    MODELS_TARGET / "syncnet"           / "latentsync_syncnet.pt",
    MODELS_TARGET / "dwpose"            / "dw-ll_ucoco_384.pth",
    MODELS_TARGET / "face-parse-bisent" / "79999_iter.pth",
    MODELS_TARGET / "face-parse-bisent" / "resnet18-5c106cde.pth",
    MODELS_TARGET / "sd-vae"            / "config.json",
    MODELS_TARGET / "sd-vae"            / "diffusion_pytorch_model.bin",
    MODELS_TARGET / "whisper"           / "config.json",
    MODELS_TARGET / "whisper"           / "preprocessor_config.json",
    MODELS_TARGET / "whisper"           / "pytorch_model.bin",
]

missing = []
for f in REQUIRED:
    actual = f.resolve() if f.is_symlink() else f
    if not actual.exists():
        missing.append(f"MISSING   : {f}")
    elif actual.stat().st_size == 0:
        missing.append(f"ZERO BYTES: {f}")

if missing:
    print("Problems found:")
    for m in missing:
        print(f"  ✗ {m}")
    raise FileNotFoundError("Fix missing files before continuing.")

print("All model files verified:")
for f in REQUIRED:
    actual = f.resolve() if f.is_symlink() else f
    size_mb = actual.stat().st_size / (1024**2)
    print(f"  ✓  {f.name:<35}  {size_mb:7.1f} MB")

## How MuseTalk models are initialized

The verification step above checks storage only. Neural networks are initialized later, when the API starts the upstream `scripts.inference` process for a `/generate` request.

| Component | Initialization location | Checkpoint/configuration | Role |
| --- | --- | --- | --- |
| Stable Diffusion VAE | `musetalk.utils.utils.load_all_model()` creates `VAE`; `VAE.__init__()` calls `AutoencoderKL.from_pretrained()` | `models/sd-vae/` | Encodes cropped face images into latent tensors and decodes generated latents back to pixels |
| MuseTalk UNet | `load_all_model()` creates `UNet`; its constructor builds `UNet2DConditionModel` and loads its state dict | `models/musetalkV15/musetalk.json` and `unet.pth` | Predicts lip-region latents conditioned on audio features |
| Positional encoding | `load_all_model()` creates `PositionalEncoding(d_model=384)` | No checkpoint | Adds temporal position information to Whisper feature chunks |
| Whisper | `scripts.inference.main()` calls `WhisperModel.from_pretrained()` | `models/whisper/` | Converts normalized audio into time-aligned conditioning features |
| DWPose | Imported `musetalk.utils.preprocessing` calls MMPose `init_model()` | DWPose config plus `dw-ll_ucoco_384.pth` | Finds facial landmarks used to determine the generated crop |
| Face detector | Initialized in `musetalk.utils.preprocessing` with `FaceAlignment` | Detector resources supplied by the dependency | Finds the face bounding box in each frame |
| Face parser | `scripts.inference.main()` creates `FaceParsing` | `79999_iter.pth` and `resnet18-5c106cde.pth` | Builds the blending mask used to merge the generated mouth region |

### Inference sequence inside `scripts.inference`

1. Load the VAE, UNet, positional encoding, Whisper and face-processing models.
2. Extract video frames and audio features.
3. Detect landmarks and crop the face region for each frame.
4. Use the VAE to encode masked and reference face crops into latents.
5. Batch audio features and image latents.
6. Run the UNet with Whisper features as conditioning.
7. Decode predicted latents through the VAE.
8. Blend generated face regions into the source frames and combine them with the driving audio.

The SyncNet checkpoint is not initialized by this generation entrypoint; it is used by other MuseTalk training/evaluation workflows.

In [ ]:
# ── STEP 8: Write sanitizer.py ────────────────────────────────────────────────
import pathlib

SANITIZER_CODE = '''\
import subprocess
from pathlib import Path


def sanitize_video(input_path: str, output_dir: str) -> str:
    out = str(Path(output_dir) / "sanitized_video.mp4")
    vf  = ("scale=512:512:force_original_aspect_ratio=decrease,"
           "pad=512:512:(ow-iw)/2:(oh-ih)/2:black")
    r = subprocess.run(
        ["ffmpeg", "-y", "-i", input_path,
         "-vf", vf, "-r", "25", "-c:v", "libx264", "-preset", "fast", "-crf", "18",
         "-an", out],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError("Video sanitisation failed: " + r.stderr)
    return out


def sanitize_audio(input_path: str, output_dir: str) -> str:
    out = str(Path(output_dir) / "sanitized_audio.wav")
    r = subprocess.run(
        ["ffmpeg", "-y", "-i", input_path,
         "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", out],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError("Audio sanitisation failed: " + r.stderr)
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", out],
        capture_output=True, text=True
    )
    if probe.returncode != 0:
        raise RuntimeError("Audio duration check failed: " + probe.stderr)
    duration = float(probe.stdout.strip())
    if duration < 0.5:
        raise ValueError("Audio too short (< 0.5 s).")
    return out
'''

pathlib.Path("/kaggle/working/sanitizer.py").write_text(SANITIZER_CODE)
print("Written: /kaggle/working/sanitizer.py")

## API request lifecycle

The generated FastAPI service contains two endpoints:

- `GET /health` checks CUDA visibility and reports free VRAM. It does not load the MuseTalk models.
- `POST /generate` accepts multipart fields named `video` and `audio` and performs the full generation workflow.

For each generation request, the service:

1. Creates an isolated temporary working directory.
2. Saves the uploaded files and records the source resolution.
3. Converts the inputs into the formats expected by MuseTalk.
4. Writes a one-task YAML inference configuration.
5. Starts `python -m scripts.inference` in the MuseTalk directory with FP16 enabled.
6. Captures process output and converts failures into API errors.
7. Finds the generated MP4, removes letterbox padding, and scales it back to the source resolution.
8. Returns the MP4 and removes request-specific temporary files.

Because inference runs in a child process, models are initialized once per request and released when that process exits. This favors isolation and simple cleanup over maximum throughput.

In [ ]:
# ── STEP 9: Write app.py ──────────────────────────────────────────────────────
import pathlib

APP_CODE = f'''
import os, subprocess, tempfile, shutil, time, traceback, json
from pathlib import Path
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import FileResponse

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
sys.path.insert(0, "/kaggle/working")
from sanitizer import sanitize_video, sanitize_audio

MUSETALK_DIR = Path("/kaggle/working/MuseTalk")
RESULT_DIR   = Path("{RESULT_DIR}")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE     = "/kaggle/working/app.log"

BATCH_SIZE = {BATCH_SIZE}
FPS        = {FPS}

app = FastAPI(title="MuseTalk 1.5 Dubbing API", version="1.0")

def log(msg: str):
    line = f"[{{time.strftime('%H:%M:%S')}}] {{msg}}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\\n")


def get_video_dimensions(path: str):
    """Return (width, height) of a video file using ffprobe."""
    r = subprocess.run(
        ["ffprobe", "-v", "error",
         "-select_streams", "v:0",
         "-show_entries", "stream=width,height",
         "-of", "json", path],
        capture_output=True, text=True
    )
    info = json.loads(r.stdout)
    stream = info["streams"][0]
    return stream["width"], stream["height"]


def restore_resolution(dubbed_512: str, orig_w: int, orig_h: int, output_path: str):
    """Scale the 512×512 dubbed video back to original resolution.

    The sanitizer letterboxed the original into 512×512:
      - landscape (w≥h): scaled to 512×scaled_h, padded top/bottom
      - portrait  (h>w): scaled to scaled_w×512, padded left/right

    We crop the padding away then scale to the original size.
    """
    if orig_w >= orig_h:
        scaled_h = round(512 * orig_h / orig_w)
        # ensure even number (required by libx264)
        scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h - 1
        pad_y = (512 - scaled_h) // 2
        crop_filter = f"crop=512:{{scaled_h}}:0:{{pad_y}}"
    else:
        scaled_w = round(512 * orig_w / orig_h)
        scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w - 1
        pad_x = (512 - scaled_w) // 2
        crop_filter = f"crop={{scaled_w}}:512:{{pad_x}}:0"

    vf = f"{{crop_filter}},scale={{orig_w}}:{{orig_h}}"
    r = subprocess.run(
        ["ffmpeg", "-y", "-i", dubbed_512,
         "-vf", vf, "-c:v", "libx264", "-preset", "fast", "-crf", "18",
         "-c:a", "copy", output_path],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError("Resolution restore failed: " + r.stderr)


@app.get("/health")
def health():
    import torch
    gpu = torch.cuda.is_available()
    return {{
        "status": "ok",
        "gpu":          torch.cuda.get_device_name(0) if gpu else "cpu",
        "vram_free_gb": round(torch.cuda.mem_get_info(0)[0] / (1024**3), 2) if gpu else None,
    }}


@app.post("/generate")
async def generate(
    video: UploadFile = File(...),
    audio: UploadFile = File(...),
):
    work_dir = tempfile.mkdtemp(dir="/kaggle/working")
    log(f"Request received | video={{video.filename}} audio={{audio.filename}} workdir={{work_dir}}")
    try:
        # 1. Save uploads
        video_raw = os.path.join(work_dir, "input_video" + Path(video.filename).suffix)
        audio_raw = os.path.join(work_dir, "input_audio" + Path(audio.filename).suffix)
        with open(video_raw, "wb") as f: f.write(await video.read())
        with open(audio_raw, "wb") as f: f.write(await audio.read())
        log(f"Files saved: {{video_raw}}, {{audio_raw}}")

        # 2. Record original resolution before sanitising
        orig_w, orig_h = get_video_dimensions(video_raw)
        log(f"Original resolution: {{orig_w}}x{{orig_h}}")

        # 3. Sanitise (downscales to 512×512 with letterbox)
        try:
            clean_video = sanitize_video(video_raw, work_dir)
            clean_audio = sanitize_audio(audio_raw, work_dir)
            log(f"Sanitised: {{clean_video}}, {{clean_audio}}")
        except (RuntimeError, ValueError) as exc:
            log(f"Sanitise FAILED: {{exc}}")
            raise HTTPException(status_code=422, detail=str(exc))

        # 4. Write inference YAML
        result_subdir = os.path.join(work_dir, "output")
        os.makedirs(result_subdir, exist_ok=True)
        yaml_path = os.path.join(work_dir, "infer.yaml")
        with open(yaml_path, "w") as f:
            f.write("task_0:\\n")
            f.write(f\'  video_path: "{{clean_video}}"\\n\')
            f.write(f\'  audio_path: "{{clean_audio}}"\\n\')
        log(f"YAML written: {{yaml_path}}")

        # 5. Run MuseTalk inference
        cmd = [
            "python", "-m", "scripts.inference",
            "--version",          "v15",
            "--unet_config",      "./models/musetalkV15/musetalk.json",
            "--unet_model_path",  "./models/musetalkV15/unet.pth",
            "--inference_config", yaml_path,
            "--result_dir",       result_subdir,
            "--batch_size",       str(BATCH_SIZE),
            "--fps",              str(FPS),
            "--use_float16",
        ]
        log(f"Running: {{' '.join(cmd)}}")
        env = {{**os.environ, "CUDA_VISIBLE_DEVICES": "0"}}
        proc = subprocess.run(cmd, cwd=str(MUSETALK_DIR), env=env, capture_output=True, text=True)

        log(f"Inference returncode: {{proc.returncode}}")
        if proc.stdout:
            log("STDOUT: " + proc.stdout[-2000:])
        if proc.stderr:
            log("STDERR: " + proc.stderr[-2000:])

        if proc.returncode != 0:
            raise HTTPException(
                status_code=500,
                detail="STDOUT:\\n" + proc.stdout[-2000:] + "\\nSTDERR:\\n" + proc.stderr[-2000:]
            )

        # 6. Find 512×512 output
        candidates = [p for p in Path(result_subdir).rglob("*.mp4") if "_concat" not in p.name]
        log(f"Output candidates: {{candidates}}")
        if not candidates:
            raise HTTPException(status_code=500, detail="No output video produced by MuseTalk.")

        dubbed_512 = str(candidates[0])

        # 7. Restore original resolution (CPU ffmpeg, fast)
        result_name = f"dubbed_{{int(time.time())}}.mp4"
        result_path = str(RESULT_DIR / result_name)
        if orig_w != 512 or orig_h != 512:
            log(f"Restoring resolution {{orig_w}}x{{orig_h}} from 512x512...")
            restore_resolution(dubbed_512, orig_w, orig_h, result_path)
        else:
            shutil.copy2(dubbed_512, result_path)

        log(f"Done → {{result_path}}")
        return FileResponse(result_path, media_type="video/mp4", filename=result_name)

    except HTTPException:
        raise
    except Exception as exc:
        log("Unhandled exception: " + traceback.format_exc())
        raise HTTPException(status_code=500, detail=traceback.format_exc())
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)
'''

pathlib.Path("/kaggle/working/app.py").write_text(APP_CODE.strip())
print("Written: /kaggle/working/app.py")

## Service exposure and lifecycle

The final step starts Uvicorn in a background thread and creates a temporary public HTTPS endpoint through ngrok. The base URL is entered in the local Gradio interface; the UI appends `/generate` when sending a request.

The ngrok token is a credential. Keep the Kaggle Dataset and notebook private while the token is present, never commit a real token, and replace it with the placeholder before sharing the notebook.

The heartbeat loop keeps the notebook process active and reports GPU memory once per minute. Interrupting the cell requests Uvicorn shutdown and closes the ngrok tunnel.

In [ ]:
# ── STEP 10: Start ngrok tunnel and FastAPI server ────────────────────────────
import threading, time, uvicorn
from pyngrok import ngrok, conf

if NGROK_AUTHTOKEN == "PASTE_YOUR_NGROK_TOKEN_HERE":
    raise ValueError(
        "Set NGROK_AUTHTOKEN in Step 1 first.\n"
        "Free token: https://dashboard.ngrok.com/get-started/your-authtoken"
    )

# Start tunnel
conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()
tunnel     = ngrok.connect(SERVER_PORT, "http")
PUBLIC_URL = tunnel.public_url

print("═" * 60)
print(f"  Public URL  : {PUBLIC_URL}")
print(f"  Health      : {PUBLIC_URL}/health")
print(f"  Generate    : {PUBLIC_URL}/generate  (POST)")
print("═" * 60)
print("Paste the Public URL into Gradio UI → Tab 2 → Kaggle API URL\n")

# Launch server in background thread
os.chdir("/kaggle/working")
config = uvicorn.Config("app:app", host="0.0.0.0", port=SERVER_PORT, log_level="info")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
print(f"Server started on port {SERVER_PORT}. Interrupt kernel to stop.\n")

# Heartbeat — keeps kernel alive, prints VRAM every 60s
import torch
try:
    while True:
        free_gb  = torch.cuda.mem_get_info(0)[0] / (1024**3)
        total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[{time.strftime('%H:%M:%S')}]  VRAM used: {total_gb - free_gb:.1f} / {total_gb:.1f} GB  |  {PUBLIC_URL}")
        time.sleep(60)
except KeyboardInterrupt:
    server.should_exit = True
    ngrok.kill()
    print("Server stopped.")